<span style="color:red">**[Required]**</span> <code>statsmodels</code>은 R과 유사하게 통계 모형을 추정해주는 Python Library입니다. **OLS 추정**을 위해 필요합니다.  
[Optional] <code>tqdm</code>은 Python의 iterable 객체의 반복 과정을 시각화하여 보여주는 Library입니다. 필수는 아닙니다.

In [ ]:
#!pip install statsmodels
#!pip install tqdm

### 라이브러리 불러오기

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm  # 사용하지 않을 시 주석처리
import statsmodels.api as sm

tqdm.pandas() # 사용하지 않을 시 주석처리

## 고객 정보 불러오기
- 사용할 변수: 나이, 성별, 구독 연수, 다셋탑 여부, IPTV 평균지출, 월정액 가입기간, 월평균 OTT 이용일수, 거주지역

In [ ]:
user_df = pd.read_csv('./user_info_306864.csv', index_col='svc_mgmt_num')

## 변수 변환
- sex_cd, mstb: 인코딩 
- svc_scrb_yr: 2023 - svc_scrb_yr (2023년 기준 구독 경과 연수)
- ppc_arpu: 표준화 (로그를 취하려 했으나 3명의 고객이 음수값을 가짐)
- 거주지역 더미변수 생성: 아래의 표와 같이 대응됩니다.

| 거주지역 (ct_pvc_nm) | 코드 | 더미변수 이름 | 비고 |
| :---: | :---: | :---: | :---: |
| 경기 | 0 | <code>ct_1</code> | |
| 서울 | 1 | <code>ct_2</code> | 기준 더미변수 |
| 경남 | 2 | <code>ct_3</code> | |
| 인천 | 3 | <code>ct_4</code> | |
| 부산 | 4 | <code>ct_5</code> | |
| 대구 | 5 | <code>ct_6</code> | |
| 경북 | 6 | <code>ct_7</code> | |
| 충남 | 7 | <code>ct_8</code> | |
| 충북 | 8 | <code>ct_9</code> | |
| 전남 | 9 | <code>ct_10</code> | |
| 전북 | 10 | <code>ct_11</code> | |
| 광주 | 11 | <code>ct_12</code> | |
| 광원 | 12 | <code>ct_13</code> | |
| 울산 | 13 | <code>ct_14</code> | |
| 대전 | 14 | <code>ct_15</code> | |
| 제주 | 15 | <code>ct_16</code> | |
| 세종 | 16 | <code>ct_17</code> | |


In [ ]:
user_df['sex_cd'] = np.where(user_df['sex_cd'] == 'M', 0, 1) # 여성이면 1, 남성이면 0
user_df['mstb_yn'] = np.where(user_df['mstb_yn'] == 'N', 0, 1) # 다셋탑이면 1, 아니면 0
user_df['svc_scrb_yr'] = 2023 - user_df['svc_scrb_yr']
user_df['ppc_arpu'] = (user_df['ppc_arpu'] - user_df['ppc_arpu'].mean()) / user_df['ppc_arpu'].std()

ct_pvc_nm_list = user_df['ct_pvc_nm'].value_counts().index.tolist()
for i, ct_pvc_nm in enumerate(ct_pvc_nm_list):
    user_df['ct_' + str(i + 1)] = np.where(user_df['ct_pvc_nm'] == ct_pvc_nm, 1, 0)

## 주 시청 시간대 추출

In [ ]:
wat_tms_rng_df = pd.read_csv('./wat_tms_rng_list.csv')

In [ ]:
wat_tms_rng_df['hday_yn_wat_tms_rng'] = wat_tms_rng_df['hday_yn_wat_tms_rng'].str.replace('[/\[|\'|\]/]', '', regex=True)

In [ ]:
def get_wat_time(x):
    num_wat = len(x.split(' '))
    num_weekend = x.count('Y')
    num_weekday = num_wat - num_weekend
    ratio_weekend = num_weekend / num_wat
    ratio_weekday = num_weekday / num_wat
    num_wh01 = x.count('01')
    num_wh02 = x.count('02')
    num_wh03 = x.count('03')
    num_wh04 = x.count('04')
    num_wh05 = x.count('05')
    num_wh06 = x.count('06')
    num_wh07 = x.count('07')
    # 아침 & 오전: WH01, WH02
    # 오후 & 저녁: WH03, WH04
    # 밤 & 심야 & 새벽: WH05, WH06, WH07
    ratio_morning = (num_wh01 + num_wh02) / num_wat
    ratio_evening = (num_wh03 + num_wh04 + num_wh05) / num_wat
    ratio_night = (num_wh06 + num_wh07) / num_wat
    return pd.Series([ratio_weekend, ratio_weekday, ratio_morning, ratio_evening, ratio_night])

In [ ]:
# tqdm 라이브러리 미사용시 progress_apply를 apply로 변경 (약 3 ~ 4분 소요)
wat_tms_rng_df[['ratio_weekday', 'ratio_weekend', 'ratio_morning', 'ratio_evening', 'ratio_night']] = wat_tms_rng_df['hday_yn_wat_tms_rng'].progress_apply(get_wat_time)

In [ ]:
wat_tms_rng_df.to_csv('wat_tms_rng_list_(added var).csv')

## 애니메이션 메타데이터, 평점데이터 및 평점 행렬 불러오기

In [ ]:
ani_df = pd.read_csv('./contents_list_2156.csv', index_col='sris_id') # 애니메이션 메타데이터
rating_raw_df = pd.read_csv('./rating_raw.csv') # 평점 데이터 (관측 단위: 샘플된 (고객, 애니메이션)의 Pair)
ani_rating_df = pd.read_csv('./ani_rating.csv') # 평점 행렬 (고객 by 애니메이션)

### 샘플 고객 ID와 Demographic 리스트, 샘플 애니메이션 리스트 추출

In [ ]:
svc_mgmt_num_list = ani_rating_df['svc_mgmt_num'].tolist() # 고객 리스트
target_ani_list = ani_rating_df.columns.tolist()[1:] # 고려되는 애니메이션 ID 리스트
user_demo_info_cols = [col for col in user_df.columns if col != 'ct_2' and col != 'ct_pvc_nm'] # 사용할 고객 Demographic 정보
user_wat_info_cols = ['ratio_weekend', 'ratio_evening', 'ratio_night'] # 사용할 고객 시청 시간대 정보

## 회귀 분석

In [ ]:
reg_result = {} # 회귀분석의 각 변수 앞에 있는 계수와 Goodness-of-Fit 수치들을 모아둘 Dictionary
for var in user_demo_info_cols + user_wat_info_cols:
    reg_result[var] = {}
    reg_result[var + '_pval'] = {}
reg_result['intercept'] = {}
reg_result['intercept_pval'] = {}
reg_result['Rsquared'] = {}
reg_result['Adj_Rsquared'] = {}
reg_result['F_pval'] = {}
reg_result['RemovedObs'] = {}

In [ ]:
for ani_id in tqdm(target_ani_list):
    removed_obs = 0

    this_ani_rating_df = rating_raw_df[rating_raw_df['sris_id'] == ani_id].reset_index(drop=True) # 해당 애니메이션 평점 정보
    this_watched_svc_mgmt = this_ani_rating_df['svc_mgmt_num'].tolist() # 해당 애니메이션에 평점이 기록된 고객들

    this_wat_time_df = wat_tms_rng_df[(wat_tms_rng_df['svc_mgmt_num'].isin(this_watched_svc_mgmt)) & (wat_tms_rng_df['sris_id'] == ani_id)][['svc_mgmt_num', 'ratio_weekend', 'ratio_evening', 'ratio_night']] # 시청시간 정보
    this_svc_mgmt_info = pd.merge(left=user_df.loc[this_watched_svc_mgmt, user_demo_info_cols], right=this_wat_time_df, on='svc_mgmt_num', how='left') # 고객정보 결합

    X_raw = this_svc_mgmt_info.drop(columns='svc_mgmt_num').to_numpy() # 고객 Demographic 및 시청 시간대 행렬 변환
    X = X_raw[~np.isnan(X_raw).any(axis=1)] # 결측치 포함시 제거
    Y = this_ani_rating_df['rating'].to_numpy()[~np.isnan(X_raw).any(axis=1)] # 종속변수(평점) 벡터
    removed_obs = X_raw.shape[0] - X.shape[0]
    results = sm.OLS(Y, sm.add_constant(X)).fit() # OLS 수행

    # 계수 저장
    for i, var in enumerate(user_demo_info_cols + user_wat_info_cols):
        reg_result[var].update({ani_id: results.params[i + 1]})
        reg_result[var + '_pval'].update({ani_id: results.pvalues[i + 1]})
    reg_result['intercept'].update({ani_id: results.params[0]})
    reg_result['intercept_pval'].update({ani_id: results.pvalues[0]})
    
    # Goodness-of-Fit 저장
    reg_result['Rsquared'].update({ani_id: results.rsquared})
    reg_result['Adj_Rsquared'].update({ani_id: results.rsquared_adj})
    reg_result['F_pval'].update({ani_id: results.f_pvalue})

    # 특이사항 저장
    reg_result['RemovedObs'].update({ani_id: removed_obs})

### 애니메이션별 회귀 분석 결과 데이터프레임 생성

In [ ]:
reg_result_df = pd.DataFrame(data=reg_result) # 데이터프레임 생성
reg_result_df.index.name = 'sris_id' # 인덱스 칼럼 이름 설정

### 애니메이션 메타데이터와 결합

In [ ]:
reg_result_df = pd.merge(left=ani_df, right=reg_result_df, on='sris_id', how='right')

### CSV 파일로 추출

In [ ]:
reg_result_df.to_csv('./571_ani_with_reg_on_user_demo_and_wat_time.csv', encoding='cp949')